# =========================================================
# Probabilistic U-Net for Image Segmentation (Colab-ready)
# Author: Alan (FYP Medical Image Segmentation)
# =========================================================

In [1]:

!pip install torch torchvision matplotlib --quiet


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------
# 1. Toy Segmentation Dataset (Random Blobs)
# ---------------------------------------------------------

In [3]:
class SyntheticBlobDataset(Dataset):
    """
    Generates simple 2D images with circular blobs.
    Task: segment out the blobs.
    """
    def __init__(self, num_samples=200, size=64):
        self.num_samples = num_samples
        self.size = size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        img = np.zeros((self.size, self.size), dtype=np.float32)
        mask = np.zeros_like(img)

        # random circle
        cx, cy = np.random.randint(15, 50, size=2)
        r = np.random.randint(5, 10)
        for x in range(self.size):
            for y in range(self.size):
                if (x - cx)**2 + (y - cy)**2 < r**2:
                    img[x, y] = 1.0
                    mask[x, y] = 1.0

        img = torch.tensor(img).unsqueeze(0)   # (1,H,W)
        mask = torch.tensor(mask).unsqueeze(0) # (1,H,W)
        return img, mask


# ---------------------------------------------------------
# 2. Standard U-Net Encoder-Decoder (Shared Backbone)
# ---------------------------------------------------------

In [4]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNetBackbone(nn.Module):
    def __init__(self, in_channels=1, base_ch=32):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base_ch)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base_ch, base_ch*2)
        self.pool2 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(base_ch*2, base_ch*4)

        self.up1 = nn.ConvTranspose2d(base_ch*4, base_ch*2, 2, stride=2)
        self.dec1 = DoubleConv(base_ch*4, base_ch*2)
        self.up2 = nn.ConvTranspose2d(base_ch*2, base_ch, 2, stride=2)
        self.dec2 = DoubleConv(base_ch*2, base_ch)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))

        d1 = self.up1(b)
        d1 = torch.cat([d1, e2], dim=1)
        d1 = self.dec1(d1)

        d2 = self.up2(d1)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        return d2

# ---------------------------------------------------------
# 3. Probabilistic U-Net (Latent + UNet + KL Divergence)
# ---------------------------------------------------------

In [5]:
class ProbUNet(nn.Module):
    def __init__(self, latent_dim=6):
        super().__init__()
        self.backbone = UNetBackbone()
        self.final_conv = nn.Conv2d(32, 1, 1)

        # Prior and Posterior networks (latent distribution)
        self.prior_net = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), # global features
        )
        self.posterior_net = nn.Sequential(
            nn.Conv2d(33, 32, 3, padding=1), # includes mask
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc_mu_prior = nn.Linear(32, latent_dim)
        self.fc_logvar_prior = nn.Linear(32, latent_dim)
        self.fc_mu_post = nn.Linear(32, latent_dim)
        self.fc_logvar_post = nn.Linear(32, latent_dim)

        self.latent_to_feat = nn.Linear(latent_dim, 32)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, mask=None):
        # UNet features
        feats = self.backbone(x)

        # Prior
        prior_feat = self.prior_net(feats).view(x.size(0), -1)
        mu_p = self.fc_mu_prior(prior_feat)
        logvar_p = self.fc_logvar_prior(prior_feat)

        # Posterior (only in training, with mask)
        if mask is not None:
            post_in = torch.cat([feats, mask], dim=1)
            post_feat = self.posterior_net(post_in).view(x.size(0), -1)
            mu_q = self.fc_mu_post(post_feat)
            logvar_q = self.fc_logvar_post(post_feat)
        else:
            mu_q, logvar_q = mu_p, logvar_p

        # Sample latent z
        z = self.reparameterize(mu_q, logvar_q)
        z_feat = self.latent_to_feat(z).unsqueeze(-1).unsqueeze(-1)
        z_feat = z_feat.expand(-1, -1, feats.size(2), feats.size(3))

        # Fuse latent with features
        out = self.final_conv(feats + z_feat)
        return torch.sigmoid(out), (mu_p, logvar_p, mu_q, logvar_q)

    def kl_divergence(self, mu_p, logvar_p, mu_q, logvar_q):
        """KL divergence between posterior and prior"""
        return 0.5 * torch.sum(
            logvar_p - logvar_q +
            (torch.exp(logvar_q) + (mu_q - mu_p) ** 2) / torch.exp(logvar_p) - 1
        )

# ---------------------------------------------------------
# 4. Training Loop
# ---------------------------------------------------------

In [6]:
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for img, mask in loader:
        img, mask = img.to(device), mask.to(device)

        optimizer.zero_grad()
        pred, (mu_p, logvar_p, mu_q, logvar_q) = model(img, mask)
        recon_loss = criterion(pred, mask)
        kl = model.kl_divergence(mu_p, logvar_p, mu_q, logvar_q) / img.size(0)
        loss = recon_loss + 1e-3 * kl  # KL weighted
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# ---------------------------------------------------------
# 5. Visualization Utility
# ---------------------------------------------------------

In [7]:
def visualize(model, dataset, device, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(12, num_samples*3))

    for i in range(num_samples):
        img, mask = dataset[i]
        img = img.unsqueeze(0).to(device)

        preds = []
        with torch.no_grad():
            for _ in range(3):  # sample 3 possible outputs
                pred, _ = model(img, None)
                preds.append(pred.cpu().squeeze().numpy())

        axes[i,0].imshow(img.cpu().squeeze(), cmap="gray")
        axes[i,0].set_title("Input")
        axes[i,1].imshow(mask.squeeze(), cmap="gray")
        axes[i,1].set_title("Ground Truth")
        axes[i,2].imshow(preds[0]>0.5, cmap="gray")
        axes[i,2].set_title("Sample 1")
        axes[i,3].imshow(preds[1]>0.5, cmap="gray")
        axes[i,3].set_title("Sample 2")

        for j in range(4):
            axes[i,j].axis("off")

    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# 6. Main Execution
# ---------------------------------------------------------

In [8]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_dataset = SyntheticBlobDataset(num_samples=200)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_dataset = SyntheticBlobDataset(num_samples=10)

    model = ProbUNet().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    for epoch in range(5):  # small epochs for demo
        loss = train(model, train_loader, optimizer, criterion, device)
        print(f"Epoch {epoch+1}, Loss: {loss:.4f}")

    # Visualization
    visualize(model, test_dataset, device)

if __name__ == "__main__":
    main()

KeyboardInterrupt: 